# USA Employment Statistics Dataset Visualisations

This notebook explores the dataset of employment statistics for the USA as of May, 2023 and presents different kinds of visualisations to gain insights into employment in fields involving AI, Data Science and Software Development.

For each visualisation, we explain:  

1. **The objective of the visualisation.**  
2. **Why that visualisation type was chosen.**
3. **What we learned from the plot about AI employment.**

In [ ]:
pip install pandas numpy "matplotlib>=3.11" openpyxl

Requirements: Python 3, pandas, numpy, matplotlib (3.11 or newer), openpyxl.

In [ ]:
# Import the libraries
import pandas as pd 
import numpy as np # numerical python
import matplotlib.pyplot as plt # maths plotting library - python plotting
from matplotlib.ticker import StrMethodFormatter

In [ ]:
if "df" not in globals():
    df = pd.read_excel("all_data_M_2023.xlsx")
df.head()

Roles on the dataset that are related to AI, Data science and software development:

- Data Scientists.
- Software Developers.
- Web Developers.

In [ ]:
bar_chart_roles = ["Web Developers", "Data Scientists", "Software Developers"]

# 1. checks : states, any industry, and chosen roles
chosen_roles_data = df[
    (df["AREA_TYPE"] == 2) # only checks states
    & (df["NAICS"] == "000000") # any industry
    & (df["OCC_TITLE"].isin(bar_chart_roles)) # only chosen roles
].copy()

chosen_roles_data["TOT_EMP"] = pd.to_numeric(chosen_roles_data["TOT_EMP"], errors="coerce") # total employed

# 2. Top 3 areas (by total employment across these roles)
top3_states = chosen_roles_data.groupby("AREA_TITLE")["TOT_EMP"].sum().nlargest(3).index

# 3. format to matplotlib diagram data input
top3_states_tuple = tuple(top3_states)
roles_means = {}
for role in bar_chart_roles:
    role_data = chosen_roles_data[chosen_roles_data["OCC_TITLE"] == role].set_index("AREA_TITLE")
    roles_means[role] = tuple(role_data.loc[top3_states, "TOT_EMP"])
roles_means

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6), layout='constrained')

res = ax.grouped_bar(roles_means, tick_labels=top3_states_tuple, group_spacing=1)
for container in res.bar_containers:
    ax.bar_label(container, padding=3, fmt='{:,.0f}')

# Y limit: tallest bar + 15% headroom so labels and legend don't collide
ymax = max(np.nanmax(values) for values in roles_means.values())
ax.set_ylim(0, ymax * 1.15)

# Thousands separators on the y-axis ticks
ax.yaxis.set_major_formatter(StrMethodFormatter('{x:,.0f}'))

# Add labels and titles.
ax.set_ylabel('Total employment')
ax.set_xlabel('State')
ax.set_title('Employment by role in the top 3 states')
ax.legend(loc='upper right', ncols=3)

# Add backgroung lines 
ax.grid(axis='y', alpha=0.3)
ax.set_axisbelow(True)

plt.show()

fig, ax = plt.subplots(figsize=(10, 6), layout='constrained')

res = ax.grouped_bar(roles_means, tick_labels=top3_states_tuple, group_spacing=1)
for container in res.bar_containers:
    ax.bar_label(container, padding=3, fmt='{:,.0f}')

# Y limit: tallest bar + 15% headroom so labels and legend don't collide
ymax = max(np.nanmax(values) for values in roles_means.values())
ax.set_ylim(0, ymax * 1.15)

# Thousands separators on the y-axis ticks
ax.yaxis.set_major_formatter(StrMethodFormatter('{x:,.0f}'))

# Add labels and titles.
ax.set_ylabel('Total employment')
ax.set_xlabel('State')
ax.set_title('Employment by role in the top 3 states')
ax.legend(loc='upper right', ncols=3)

# Add backgroung lines 
ax.grid(axis='y', alpha=0.3)
ax.set_axisbelow(True)

plt.show()

In [ ]:
bubble_chart_roles = ["Web Developers", "Data Scientists", "Software Developers"]
employment_states = df[
    (df["AREA_TYPE"] == 2)
    & (df["NAICS"] == "000000")
    & (df["O_GROUP"] == "detailed")
    & (df["OCC_TITLE"].isin(bubble_chart_roles))
].copy()

employment_states["TOT_EMP"] = pd.to_numeric(employment_states["TOT_EMP"], errors="coerce")

highest_employment_state = employment_states.groupby("OCC_TITLE")["TOT_EMP"].idxmax()
employment_states.loc[highest_employment_state, ["OCC_TITLE", "AREA_TITLE", "TOT_EMP"]]

In [ ]:
bubble_chart_state = "California"
# Web Developers, Data Scientists and Software Developers
total_employment_state = [10590, 33220, 304390]

# bubble sizes: the biggest value gets 10,000, the others are scaled to match
sizes = [n / max(total_employment_state) * 10000 for n in total_employment_state]

fig, ax = plt.subplots(figsize=(10, 4))
ax.scatter([1, 2, 3], [0, 0, 0], s=sizes, c=["#4C72B0", "#DD8452", "#55A868"], alpha=0.8)

# role name and number under each bubble
for x, role, n in zip([1, 2, 3], bubble_chart_roles, total_employment_state):
    ax.text(x, -0.5, f"{role} \n in {bubble_chart_state}:\n{n:,}", ha="center", va="top")

ax.set_xlim(0.4, 3.6)
ax.set_ylim(-1, 0.8)
ax.axis("off")   # hide the axes, the position of the bubbles has no meaning
ax.set_title(f"Highest employed state by role")
plt.show()

# Each role's Highest employed state 

**Visualisation's goal**

As discussed on the previous chart, even though it was size biased by the software developers role, when checking of each role's highest employed state individually, California remains the highest employed state on all of them, consequently having the same results as software developers being the most employed role. As well as the values remain the exact same, showing the reliability of the samples. Furthermore we learned that California is a hub that concentrates most IT roles in the US, at least the between those three roles.


**Why that visualisation type was chosen ?**

This bubble chart gives a visualisation focusing on the translation of the numbers to the sizes of each bubble to provide a visual understanding of how big is the difference of employment of each role in its respectives highest employed states.

**What you learned about AI employment from the visualisation?**
California is the hub for IT roles, consequently the AI hub in the US, or at least one of them.

In [ ]:
pie_chart_roles = ["Web Developers", "Data Scientists", "Software Developers"]
# Data Scientists, Software Developers and Web Developers total employment in the US
total_employment_chosen_roles = df[
    (df["OCC_TITLE"].isin(pie_chart_roles))
    & (df["AREA_TYPE"] == 1)
    & (df["NAICS"] == "000000")
    & (df["O_GROUP"] == "detailed")
].copy()

total_employment_chosen_roles["TOT_EMP"] = pd.to_numeric(total_employment_chosen_roles["TOT_EMP"], errors="coerce")

total_employment_chosen_roles.set_index("OCC_TITLE")["TOT_EMP"]

In [ ]:
#Pie chart of the comparison between the three chosen roles total employment in the US

# Web Developers, Data Scientists and Software Developers 
US_total_employed = [85350, 192710, 1656880]
legend_labels = [f"{role}: {n:,}" for role, n in zip(pie_chart_roles, US_total_employed)]

fig, ax = plt.subplots()
wedges, texts, autotexts = ax.pie(US_total_employed, autopct='%1.1f%%', startangle=90)


ax.legend(wedges, legend_labels, title="Total employment", loc='center left', bbox_to_anchor=(1, 0.5))
ax.set_title('National total employment on each role')
plt.show()

# Highest employed role between the three selected in the US

**Goal of the visualisation:**

This visualisation shows that on national level, software developers dominates the industry compared to the other two roles, and that even though the top 3 states chart was size biased, software developers are still the most employed role on this comparison.  

**Why that visualisation type was chosen?** 

Pie chart is good to compare the size of each role slice, also it increments the visualisation with percentages giving a better understanding of how much each role takes on the market on this case. 

**What you learned about AI employment from the visualisation ?**

Assuming that Data Scientist is the main AI role, currently it is not the highest employed, but it probably was greatly increased on recent years, to the point that is the second biggest slice on the chart, also software developers can be not directly on the development of AI, but on support roles on the development of the AI software, as well as those roles are affected by the use of AI tools.

In [ ]:
# Check each role's payrate in California
range_chart_roles = ["Web Developers", "Data Scientists", "Software Developers"]
range_chart_state = "California"
pay_ca = df[
    (df["AREA_TITLE"] == range_chart_state)
    & (df["NAICS"] == "000000")
    & (df["O_GROUP"] == "detailed")
    & (df["OCC_TITLE"].isin(range_chart_roles))
][["OCC_TITLE", "A_PCT10", "A_MEDIAN", "A_PCT90", "A_MEAN"]]

pay_ca

In [ ]:
p10 = [50440, 68050, 103240]      # A_PCT10 values
median = [111910, 133690, 168660]   # A_MEDIAN values
p90 = [204960, 213080, 226670]      # A_PCT90 values

fig, ax = plt.subplots(figsize=(9, 4))
ax.hlines(y=range_chart_roles, xmin=p10, xmax=p90, color="gray", linewidth=4, alpha=0.6)
ax.plot(median, range_chart_roles, "o", markersize=12, color="#DD8452", label="Median")
ax.plot(p10, range_chart_roles, "|", markersize=15, color="gray")
ax.plot(p90, range_chart_roles, "|", markersize=15, color="gray")

for x, y in zip(median, range_chart_roles):
    ax.text(x, y, f"${x:,}", ha="center", va="bottom")

ax.set_xlabel("Annual pay in Dollars")
ax.set_title("Annual pay by role in California (10th to 90th percentile, median marked)")
ax.legend()
ax.invert_yaxis()
plt.show()

# Annual salaries in California

**Visualisation goal**

This visualisation compares the annual pay of each role in California, the most employed state of those roles in the US, to find out that the highest paid role is Software developer, with the annual 10th percentile almost the same to the median of web developers with a difference of ten thousand dollars roughly and close to the median of Data Scientists as well, but with a bit bigger difference of aproximately thirty thousand dollars. As well as we are able to note the range of payment in each role, with difference of 150k dollars between the maximum and minimum.


**Why that visualisation type was chosen**

The range chart provides a visualisation of difference between entry level and senior level payment on the same role, shows the median salary of it, providing an understanding of the current payment possibilities and limits of those roles. Furthermore it enables the understanding that the pay of those roles, specially web developers, is right-skewed with the median on the left of the centre, meaning that the pay is not evenly spread and that few high earners pull the range up.


**What you learned about AI employment from the visualisation**

Even though software developers are way more employed, data scientist's annual pay median is not too far off from software developer's median.